# RPO Runner — SEC 全量 10-K / RPO 抓取管线

流程：
1. 拉取公司 CIK 列表——来自 WRDS/Compustat 的 gvkey↔cik 对照表（见 `SAS.txt` 产出的 `gvkey_cik_fyear.csv`，`gvkey_universe.py` 负责加载和清洗），而不是 SEC 全量 ticker 列表
2. 对每个 CIK 抓取 2013–2025 年的 10-K HTML 链接（`html_generator.py`），并把 `gvkey` 挂回每一行
3. 对每个 10-K 链接抓取 XBRL instance，提取 Remaining Performance Obligation（RPO）相关 tag（`XBRL.py`）

支持断点续跑：中途中断后重新运行同一个 cell，会自动跳过已经完成的部分。

**先用 `TEST_MODE = True` 跑几家公司验证一遍，再切换成 `False` 跑全量。**
全量运行前，需要先在 WRDS 上跑 `SAS.txt`，把导出的 `gvkey_cik_fyear.csv` 放进本项目文件夹。
全量大约是 Compustat 覆盖的全部公司（数千家），2013–2025 逐年 10-K，
预计会产生数万次 HTTP 请求，运行时间会很长（数小时到更久），请耐心等待，
且中途可以随时中断，重跑同一个 cell 即可继续。

In [ ]:
from gvkey_universe import get_gvkey_cik_map
from html_generator import export_company_cik_year_html
from XBRL import run_rpo_pipeline

# ========= 运行配置 =========
TEST_MODE = True  # 先用 True 验证；确认后改为 False 跑全量
START_YEAR = 2013
END_YEAR = 2025
INCLUDE_AMENDED_10K = False
MAX_WORKERS = 6
RESUME = True

GVKEY_CIK_CSV = "gvkey_cik_fyear.csv"  # SAS.txt 在 WRDS 上跑出来的产出，放在本项目文件夹

# 美国公司的最终定义还没确定，所以默认不强制筛选。
# 设为 True 时，先按 SEC stateOfIncorporation 保留 50 州 + DC 注册公司。
APPLY_US_INCORPORATION_FILTER = False
US_STATE_CODES = {
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
    'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
    'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
    'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
    'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'DC',
}

# 后续如果要按 SIC、交易所或其他条件筛选，把下面改成一个函数：
# CUSTOM_FILING_FILTER = lambda df: df[df['exchanges'].str.contains('Nasdaq|NYSE', na=False)]
CUSTOM_FILING_FILTER = None

## Step 0 — 获取 CIK 列表（来自 Compustat gvkey↔cik）

`get_gvkey_cik_map` 读取 `gvkey_cik_fyear.csv`（`SAS.txt` 在 WRDS 上跑出来的），
清洗 cik（去掉缺失/非数字的行，补零成 10 位），按 cik 去重成公司级的 gvkey↔cik 对照表。
`gvkey` 会一路带到 Step 1 的输出里，方便之后按 gvkey+fyear 和 Compustat 财务数据合并。

In [ ]:
import os
import pandas as pd

if os.path.exists(GVKEY_CIK_CSV):
    gvkey_cik_map = get_gvkey_cik_map(GVKEY_CIK_CSV)
    print(f"{GVKEY_CIK_CSV} 共 {len(gvkey_cik_map)} 家公司（去重后的 gvkey-cik）")
elif TEST_MODE:
    # 还没在 WRDS 上跑出 gvkey_cik_fyear.csv 也没关系，TEST_MODE 用固定的几家公司验证管线，
    # 用不到这张表；等文件放进项目文件夹后，全量运行（TEST_MODE=False）会自动用上。
    print(f"没找到 {GVKEY_CIK_CSV}（TEST_MODE 下不需要它，跳过）")
    gvkey_cik_map = pd.DataFrame(columns=["gvkey", "cik"])
else:
    raise FileNotFoundError(
        f"找不到 {GVKEY_CIK_CSV}。先在 WRDS 上跑 SAS.txt，把导出的 gvkey_cik_fyear.csv 放进本项目文件夹。"
    )

gvkey_cik_map.head()

In [ ]:
if TEST_MODE:
    # 跑几家熟悉的公司验证管线（Oracle / Apple / Tesla / Microsoft）
    cik_ls = ["0001341439", "0000320193", "0001318605", "0000789019"]
else:
    cik_ls = gvkey_cik_map["cik"].tolist()

print(f"本次将抓取 {len(cik_ls)} 家公司")

## Step 1 — 抓取每家公司 2013–2025 年 10-K 的 HTML 链接

`export_company_cik_year_html` 会：
- 按 CIK 抓 SEC submissions API，筛选 10-K（可选 include_amended=True 把 10-K/A 也算进来）
- 顺带记录 `stateOfIncorporation` / `sic` / `sicDescription` / `exchanges`，方便之后按"美股/美国公司"等条件筛选
- 每处理 `checkpoint_every` 家公司自动存盘一次；重跑时 `resume=True` 会跳过已完成的 CIK

In [ ]:
html_out_xlsx = (
    f"company_cik_fy_html_10k_{START_YEAR}_{END_YEAR}_test.xlsx"
    if TEST_MODE else
    f"company_cik_fy_html_10k_{START_YEAR}_{END_YEAR}_full.xlsx"
)

df_html = export_company_cik_year_html(
    cik_ls,
    out_xlsx=html_out_xlsx,
    start_year=START_YEAR,
    end_year=END_YEAR,
    include_amended=INCLUDE_AMENDED_10K,
    max_workers=MAX_WORKERS,
    checkpoint_every=5 if TEST_MODE else 50,
    resume=RESUME,
)

# 把 gvkey 挂回每一行（按 cik 匹配 Compustat 的公司级映射），方便后面按 gvkey 合并 Compustat 财务数据
df_html = df_html.merge(gvkey_cik_map, left_on="CIK", right_on="cik", how="left").drop(columns=["cik"])

print(f"HTML filing 清单：{len(df_html):,} 条 -> {html_out_xlsx}")
df_html.head()

## Step 1.5 — 可配置的公司 / filing 筛选

默认保留所有 10-K。可以在顶部打开美国注册地筛选，或提供 `CUSTOM_FILING_FILTER` 函数。筛选只影响后面的 XBRL 抓取，完整 HTML 清单仍会保留在 Excel 中。

In [ ]:
df_filings = df_html.copy()

if APPLY_US_INCORPORATION_FILTER:
    before = len(df_filings)
    state = df_filings['stateOfIncorporation'].fillna('').str.upper().str.strip()
    df_filings = df_filings[state.isin(US_STATE_CODES)].copy()
    print(f"美国注册地筛选：{before:,} -> {len(df_filings):,} filings")

if CUSTOM_FILING_FILTER is not None:
    before = len(df_filings)
    df_filings = CUSTOM_FILING_FILTER(df_filings.copy())
    print(f"自定义筛选：{before:,} -> {len(df_filings):,} filings")

if df_filings.empty:
    raise ValueError("筛选后没有 filing，请检查筛选条件。")

df_filings = df_filings.drop_duplicates(subset=['html']).reset_index(drop=True)
print(f"将进入 XBRL 提取：{len(df_filings):,} 条 10-K")

## Step 2 — 对每条 10-K 链接抓取 XBRL，提取 RPO

`run_rpo_pipeline` 会：
- 定位每个 filing 目录下真正的 XBRL instance XML
- 提取包含 `RemainingPerformanceObligation` 的数值 tag，以及 TextBlock 里出现 "remaining performance obligation" 的段落
- 每处理 `checkpoint_every` 条 filing 自动存盘一次（xlsx + csv）；重跑时 `resume=True` 会跳过已处理的 html_url

In [ ]:
# 全量运行直接写入你现有的 rpo_new_xbrl_output.xlsx / .csv 命名系列
rpo_output_prefix = "rpo_new_xbrl_output_test" if TEST_MODE else "rpo_new_xbrl_output"

df_rpo = run_rpo_pipeline(
    df=df_filings,
    output_prefix=rpo_output_prefix,
    max_workers=MAX_WORKERS,
    checkpoint_every=10 if TEST_MODE else 100,
    resume=RESUME,
)

print(f"RPO 结果：{len(df_rpo):,} 条 -> {rpo_output_prefix}.xlsx / .csv")
df_rpo.head()

## 快速检查结果

In [ ]:
summary = (
    df_rpo.assign(has_rpo=df_rpo["num_rpo_tags"] > 0)
    .groupby("Company")["has_rpo"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "filings_with_rpo", "count": "total_filings"})
)
summary